# MRI processing

## ============================================================================
## SECTION 1: Setup and Data Import
## ============================================================================

In [ ]:
## ============================================================================
## SECTION 1.1: Import Libraries
## ============================================================================
# All libraries are used in the script:
# - numpy, pandas: data manipulation
# - os, fnmatch: file system operations
# - nibabel: NIfTI file I/O
# - matplotlib.pyplot: visualization (optional, used in exploration)
# - scipy.stats.iqr: interquartile range calculation
# - joblib: parallel processing
# - logging: logging functionality
import numpy as np 
import pandas as pd 
import os, fnmatch
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.stats import iqr

from joblib import Parallel, delayed
import logging

In [ ]:
## OPTIONAL: Display options (not needed for pipeline)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 30)

In [ ]:
## ============================================================================
## SECTION 1.2: Load Metadata
## ============================================================================
# Load shuffled metadata file
# Old code (commented out): create shuffled version from original
meta2 = pd.read_csv('metafile_completing/metafile_completed_ADNI_A4_processed_02_06_2024_shuffled.csv')


In [ ]:
## EXPLORATION CODE: Count values by Modality (not needed for pipeline)
# meta2['Modality'].value_counts()

MRI    42804
PET    22476
Name: Modality, dtype: int64

In [ ]:
## ============================================================================
## SECTION 2: Filter by Modality
## ============================================================================
## SECTION 2.1: Select MRI and Amyloid PET Scans
# Keep: All MRI scans + AV45 PET scans
# Other tracers (FBB, PIB) commented out
pd.set_option('display.max_rows', 20)
meta3 = meta2[(meta2.Modality == 'MRI') | 
      (meta2.modality_subtype == 'AV45')]#|
#     (meta2.modality_subtype == 'FBB')|
#     (meta2.modality_subtype == 'PIB')]

In [ ]:
## UNUSED CODE: Old filtering method (commented out, using mask-based approach instead)
# selecting only T1 and amyloid PET scans with needed pre-proccesing 
'''meta4 = meta3[((meta3.Project == 'ADNI')&((meta3.Description.str.contains('MP*RAGE|T1|IR*SPGR')) | 
     (meta3.Description.str.contains('Co-registered, Averaged'))))|
           (( meta3.Project == 'A4')&((meta3.modality_subtype.str.contains('T1')) | 
     (meta3.Description.str.contains('Flor*'))))]'''

"meta4 = meta3[((meta3.Project == 'ADNI')&((meta3.Description.str.contains('MP*RAGE|T1|IR*SPGR')) | \n     (meta3.Description.str.contains('Co-registered, Averaged'))))|\n           (( meta3.Project == 'A4')&((meta3.modality_subtype.str.contains('T1')) | \n     (meta3.Description.str.contains('Flor*'))))]"

In [ ]:
## ============================================================================
## SECTION 2.2: Create Masks for Filtering
## ============================================================================
# Create boolean masks for PET and MRI scans with specific processing/sequences
pet_mask = ((meta3.Modality == 'PET')&(meta3.Description.str.contains('Co-registered, Averaged')|
            meta3.Description.str.contains('^Flor.*')))
mri_mask = (meta3.Modality == 'MRI')&(meta3.Description.str.contains('MP.*RAGE|T1|IR.*SPGR')|
            meta3.modality_subtype.str.contains('T1'))

In [ ]:
## ============================================================================
## SECTION 2.3: Apply Filters
## ============================================================================
# Filter metadata using PET and MRI masks
meta4 = meta3[pet_mask|mri_mask]
meta4.reset_index(drop=True, inplace = True)

In [ ]:
## ============================================================================
## SECTION 3: Extract Year from Study Date
## ============================================================================
# Extract year from Study.Date column for temporal analysis
meta4 = meta4.copy()
meta4['year'] = ''
for i in range(meta4.shape[0]):
    meta4['year'].values[i] = meta4['Study.Date'].values[i].split('/')[2]

In [ ]:
## ============================================================================
## SECTION 4: Separate PET and MRI Metadata
## ============================================================================
# Split metadata into PET and MRI dataframes for pairing
pet_meta = meta4[meta4.Modality == 'PET']
pet_meta.reset_index(drop=True, inplace = True)

mri_meta = meta4[meta4.Modality == 'MRI']
mri_meta.reset_index(drop=True, inplace = True)



In [ ]:
## ============================================================================
## SECTION 5: Create PET-MRI Pairing Table
## ============================================================================
## SECTION 5.1: Initialize Table Structure
# Create table starting with PET metadata and add empty columns for MRI data
pet_mri_tab = pet_meta[['Subject.ID','Project','Phase' ,'Sex','Weight','Research.Group','VISCODE','Study.Date','Age','Modality','Description','Imaging.Protocol','Image.Data.ID','modality_subtype','PATH']]
pet_mri_tab = pet_mri_tab.copy()
pet_mri_tab['MRI_subID']= ''
pet_mri_tab['MRI_Modality']= ''
pet_mri_tab['MRI_Research.Group']= ''
pet_mri_tab['MRI_Age']= np.nan
pet_mri_tab['MRI_VISCODE']= ''
pet_mri_tab['MRI_studydate']= ''
pet_mri_tab['MRI_Research.Group']= ''
pet_mri_tab['MRI_Description']= ''
pet_mri_tab['MRI_Imaging.protocol']= ''
pet_mri_tab['MRI_Type']= ''
pet_mri_tab['MRI_ID']= ''
pet_mri_tab['MRI_PATH']= ''

In [ ]:
## ============================================================================
## SECTION 5.2: Match PET Scans with MRI Scans
## ============================================================================
# Main pairing algorithm: For each PET scan, find best matching MRI scan
# Matching criteria (in order of preference):
#   1. Same VISCODE
#   2. Same Age
#   3. Age difference <= 1 year
# For ADNI 3 phase, prefer Original over Processed MRI scans
suc = 0 
er = 0

#for i in range(1992):
for i in range(pet_meta.shape[0]):
    pet_id = pet_meta.iloc[i,:]
    
    mri_data = mri_meta.copy()
    
    mri_data= mri_data[mri_data['Subject.ID']==pet_id['Subject.ID']]
                        
    mri_data['age_diff'] = abs(float(pet_id.Age) - mri_data.Age.astype(float))
    
    mri_data= mri_data[((mri_data['Age'] == pet_id['Age'])
                        | (mri_data['VISCODE'] == pet_id['VISCODE'])
                        |(abs(mri_data.age_diff)<=1))]
                          
    if mri_data.shape[0]== 0:
        er += 1
        del mri_data
        print(str(i) + ' '+ pet_id['Subject.ID']+' error')
        continue
        
    ideal_mri = mri_data[mri_data['VISCODE'] == pet_id['VISCODE']]
    if ideal_mri.shape[0] == 0:
        ideal_mri = mri_data[mri_data['Age'] == pet_id['Age']]
    if ideal_mri.shape[0] == 0:
        ideal_mri = mri_data[mri_data.age_diff == mri_data.age_diff.min()]
    if ideal_mri.shape[0] == 0:
        er += 1
        del mri_data
        print(str(i) + ' '+ pet_id['Subject.ID']+' error')
        continue
            
    mri_data_orig = ideal_mri[ideal_mri['Type'] == 'Original']
    mri_data_proc = ideal_mri[ideal_mri['Type'] == 'Processed']
    
    if pet_id['Phase'] == 'ADNI 3' and mri_data_orig.shape[0]>0:
        suc += 1
        pet_mri_tab['MRI_subID'].values[i]= mri_data_orig.iloc[0,:]['Subject.ID']
        pet_mri_tab['MRI_Modality'].values[i]= mri_data_orig.iloc[0,:]['Modality']
        pet_mri_tab['MRI_Research.Group'].values[i]= mri_data_orig.iloc[0,:]['Research.Group']
        pet_mri_tab['MRI_Age'].values[i]= mri_data_orig.iloc[0,:]['Age']
        pet_mri_tab['MRI_VISCODE'].values[i]= mri_data_orig.iloc[0,:]['VISCODE']
        pet_mri_tab['MRI_studydate'].values[i]= mri_data_orig.iloc[0,:]['Study.Date']
        pet_mri_tab['MRI_Description'].values[i]= mri_data_orig.iloc[0,:]['Description']
        pet_mri_tab['MRI_Imaging.protocol'].values[i]= mri_data_orig.iloc[0,:]['Imaging.Protocol']
        pet_mri_tab['MRI_Type'].values[i]= mri_data_orig.iloc[0,:]['Type']
        pet_mri_tab['MRI_ID'].values[i]= mri_data_orig.iloc[0,:]['Image.Data.ID']
        pet_mri_tab['MRI_PATH'].values[i]= mri_data_orig.iloc[0,:]['PATH']

    elif ((pet_id['Phase'] != 'ADNI 3' and mri_data_proc.shape[0]>0) or 
          (pet_id['Project'] == 'A4' and mri_data_proc.shape[0]>0)):
        suc += 1
        pet_mri_tab['MRI_subID'].values[i]= mri_data_proc.iloc[0,:]['Subject.ID']
        pet_mri_tab['MRI_Modality'].values[i]= mri_data_proc.iloc[0,:]['Modality']
        pet_mri_tab['MRI_Research.Group'].values[i]= mri_data_proc.iloc[0,:]['Research.Group']
        pet_mri_tab['MRI_Age'].values[i]= mri_data_proc.iloc[0,:]['Age']
        pet_mri_tab['MRI_VISCODE'].values[i]= mri_data_proc.iloc[0,:]['VISCODE']
        pet_mri_tab['MRI_studydate'].values[i]= mri_data_proc.iloc[0,:]['Study.Date']
        pet_mri_tab['MRI_Description'].values[i]= mri_data_proc.iloc[0,:]['Description']
        pet_mri_tab['MRI_Imaging.protocol'].values[i]= mri_data_proc.iloc[0,:]['Imaging.Protocol']
        pet_mri_tab['MRI_Type'].values[i]= mri_data_proc.iloc[0,:]['Type']
        pet_mri_tab['MRI_ID'].values[i]= mri_data_proc.iloc[0,:]['Image.Data.ID']
        pet_mri_tab['MRI_PATH'].values[i]= mri_data_proc.iloc[0,:]['PATH']


In [23]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_Age'].notna()]

In [24]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['Age']>0]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [25]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_Type'] == 'Processed']
pet_mri_tab.reset_index(drop=True, inplace = True)

In [27]:
if (pet_mri_tab['Subject.ID'] == pet_mri_tab['MRI_subID']).all():
    print("✅ All Subject.ID values match MRI_subID.")
else:
    print("❌ There are mismatches between Subject.ID and MRI_subID.")

✅ All Subject.ID values match MRI_subID.


In [333]:
# add path to pet scans normalised to whole cerebellum, script cerebellum_normalisation. 
for i in range(0,np.shape(pet_mri_tab)[0]):
    name_pet = pet_mri_tab['Image.Data.ID'][i]+'_normalised.nii'
    filename = fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_normalised_pet/'), name_pet)
    if len(filename)> 0:
        pet_mri_tab.loc[i,'PET_PATH_normalised'] = '/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_normalised_pet/' + filename[0]
    
    name_mri = pet_mri_tab['MRI_ID'][i]+'_registered.nii'
    filename = fnmatch.filter(os.listdir('/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_mri/'), name_mri)
    if len(filename)> 0:
        pet_mri_tab.loc[i,'MRI_PATH_registered'] = '/csc/epitkane/data/ADNI_A4/ADNI_16_04_22_A4_25_10_23_registered_mri/' + filename[0]
     
    #print(i)

In [117]:
pet_mri_tab = pet_mri_tab[pet_mri_tab['PET_PATH_normalised'].notna()]
pet_mri_tab.reset_index(drop=True, inplace = True)
pet_mri_tab = pet_mri_tab[pet_mri_tab['MRI_PATH_registered'].notna()]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [52]:
# remove scans with bad registration
meta_qc = pd.read_csv('/csc/epitkane/home/atagmazi/ADDL_pipeline/scripts/registration/metafile_ADDLpipeline_abeta_mri_27_05_2025_afterQC.csv',header=[0], index_col=[0])
pet_mri_tab = pet_mri_tab[pet_mri_tab['Image.Data.ID'].isin(meta_qc['Image.Data.ID'])].copy()
pet_mri_tab.reset_index(drop=True, inplace = True)

In [ ]:
#split data on train and test subsets
train_size = 0.8
train_end = int(len(pet_mri_tab)*train_size)
t = int(0.8*train_end) #!!
v = int(0.2*train_end) 
#DATADIR = r"/csc/epitkane/data/ADNI/AD_DL_03_11_2021/"
print(f'train set size = {t}')

In [ ]:
'''pet_mri_tab['pet_min'] = 0
pet_mri_tab['pet_max'] = 0
pet_mri_tab['pet_95q'] = 0
pet_mri_tab['pet_99q'] = 0

pet_mri_tab['mri_min'] = 0
pet_mri_tab['mri_max'] = 0
pet_mri_tab['mri_95q'] = 0
pet_mri_tab['mri_99q'] = 0


i_invalid = []
pet_all_values = []
mri_all_values = []
for i in range(pet_mri_tab.shape[0]):
    img_pet = nib.load(pet_mri_tab['PET_PATH_normalised'][i])
        
    img_mri = nib.load(pet_mri_tab['MRI_PATH_registered'][i])
    
    image_pet = np.asarray(img_pet.get_fdata())
    image_mri = np.asarray(img_mri.get_fdata())
    
    if np.isnan(image_pet).any() or np.isnan(image_mri).any():
        print(f"Skipping index {i}: NaNs detected in PET or MRI scan.")
        i_invalid.append(i)
    elif np.max(image_pet) <= 0 or np.max(image_mri) <= 0:  # Changed from == 0 to <= 0
        print(f"Skipping index {i}: Empty or non-positive PET or MRI scan.")
        i_invalid.append(i)
    else:
        pet_mri_tab.loc[i,'pet_min'] = np.min(image_pet)
        pet_mri_tab.loc[i,'pet_max'] = np.max(image_pet)
        pet_mri_tab.loc[i,'pet_99q'] = np.percentile(image_pet, 99)
        pet_mri_tab.loc[i,'pet_999q'] = np.percentile(image_pet, 99.9)
        
        # Store min & max values (ensures outliers are captured)
        #pet_all_values.append(image_pet.min())
        while i<=t:
            pet_all_values.append(image_pet.max())
        # Sample a subset of pixels (reduces memory usage)
            pet_all_values.extend(np.random.choice(image_pet.flatten(), size=min(10000, image_pet.size), replace=False))
        #pet_all_values.extend(image_pet[image_pet > 0].flatten())


        
        pet_mri_tab.loc[i,'mri_min'] = np.min(image_mri)
        pet_mri_tab.loc[i,'mri_max'] = np.max(image_mri)
        pet_mri_tab.loc[i,'mri_99q'] = np.percentile(image_mri, 99)
        pet_mri_tab.loc[i,'mri_999q'] = np.percentile(image_mri, 99.9)
        # Store min & max values (ensures outliers are captured)
        #mri_all_values.append(image_mri.min())
        while i <=t:
            mri_all_values.append(image_mri.max())
        # Sample a subset of pixels (reduces memory usage)
            mri_all_values.extend(np.random.choice(image_mri.flatten(), size=min(10000, image_mri.size), replace=False))
        #mri_all_values.extend(image_mri[image_mri > 0].flatten())

        
    print(i)
    
print(i_invalid)    
pet_minimum = np.min(pet_mri_tab['pet_min'])
pet_maximum = np.max(pet_mri_tab['pet_max'])
mri_minimum = np.min(pet_mri_tab['mri_min'])
mri_maximum = np.max(pet_mri_tab['mri_max'])
print(f'MinMax computation is done! PET minimum is {pet_minimum}, PET maximum is {pet_maximum}. MRI minimum is {mri_minimum}, MRI maximum is {mri_maximum}')
'''

/tmp/ipykernel_43436/1921735778.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.908689022064209' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_max'] = np.max(image_pet)
/tmp/ipykernel_43436/1921735778.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.7029719293117513' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_95q'] = np.percentile(image_pet, 95)
/tmp/ipykernel_43436/1921735778.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.2386636161804194' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'pet_99q'] = np.percentile(image_pet, 99)
/

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52


/tmp/ipykernel_43436/1921735778.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.8371753692626953' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pet_mri_tab.loc[i,'mri_min'] = np.min(image_mri)


53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
1861
1862
1863
1864
1865
1866
1867
1868
1869
1870
1871
1872
1873
1874
1875
1876
1877
1878
1879
1880
1881
1882
1883
1884
1885
1886
1887
1888
1889
1890
1891
1892
1893
1894
1895
1896
1897
1898
1899
1900
1901
1902
1903
1904
1905
1906
1907
1908
1909
1910
1911
1912
1913
1914
1915
1916
1917
1918
1919
1920
1921
1922
1923
1924
1925
1926
1927
1928
1929
1930
1931
1932
1933
1934
1935
1936
1937
1938
1939
1940
1941
1942
1943
1944
1945
1946
1947
1948
1949
1950
1951
1952
1953
1954
1955
1956
1957
1958
1959
1960
1961
1962
1963
1964
1965
1966
1967
1968
1969
1970
1971
1972
1973
1974
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
200

In [ ]:

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)
def process_row(i, pet_path, mri_path):
    result = {
        'i': i,
        'pet_min': 0,
        'pet_max': 0,
        'pet_90q': 0,
        'pet_95q': 0,
        'pet_99q': 0,
        'pet_999q': 0,
        'mri_min': 0,
        'mri_max': 0,
        'mri_90q': 0,
        'mri_95q': 0,
        'mri_99q': 0,
        'mri_999q': 0,
        'pet_values': [],
        'mri_values': [],
        'invalid': False
    }
    logger.info(f"Processing index {i}")  # Log each index being processed

    try:
        image_pet = np.asarray(nib.load(pet_path).get_fdata())
        image_mri = np.asarray(nib.load(mri_path).get_fdata())
        
        if np.isnan(image_pet).any() or np.isnan(image_mri).any():
            result['invalid'] = True
            return result
        elif np.max(image_pet) <= 0 or np.max(image_mri) <= 0:
            result['invalid'] = True
            return result
        
        result['pet_min'] = np.min(image_pet)
        result['pet_max'] = np.max(image_pet)
        result['pet_90q'] = np.percentile(image_pet, 90)
        result['pet_95q'] = np.percentile(image_pet, 95)
        result['pet_99q'] = np.percentile(image_pet, 99)
        result['pet_999q'] = np.percentile(image_pet, 99.9)
        if i <= t:
            result['pet_values'] = np.random.choice(image_pet.flatten(), size=min(10000, image_pet.size), replace=False)
        
        result['mri_min'] = np.min(image_mri)
        result['mri_max'] = np.max(image_mri)
        result['mri_90q'] = np.percentile(image_mri, 90)
        result['mri_95q'] = np.percentile(image_mri, 95)
        result['mri_99q'] = np.percentile(image_mri, 99)
        result['mri_999q'] = np.percentile(image_mri, 99.9)
        if i <= t:
            result['mri_values'] = np.random.choice(image_mri.flatten(), size=min(10000, image_mri.size), replace=False)
        
    except Exception as e:
        print(f"Error processing index {i}: {e}")
        result['invalid'] = True
    
    return result

# Number of workers to run in parallel (adjust based on your CPU)
n_jobs = 8

results = Parallel(n_jobs=n_jobs)(
    delayed(process_row)(i, pet_mri_tab['PET_PATH_normalised'][i], pet_mri_tab['MRI_PATH_registered'][i])
    for i in range(pet_mri_tab.shape[0])
)

# Rebuild dataframe from results
for res in results:
    i = res['i']
    if not res['invalid']:
        pet_mri_tab.loc[i, 'pet_min'] = res['pet_min']
        pet_mri_tab.loc[i, 'pet_max'] = res['pet_max']
        pet_mri_tab.loc[i, 'pet_90q'] = res['pet_90q']
        pet_mri_tab.loc[i, 'pet_95q'] = res['pet_95q']
        pet_mri_tab.loc[i, 'pet_99q'] = res['pet_99q']
        pet_mri_tab.loc[i, 'pet_999q'] = res['pet_999q']
        
        pet_mri_tab.loc[i, 'mri_min'] = res['mri_min']
        pet_mri_tab.loc[i, 'mri_max'] = res['mri_max']
        pet_mri_tab.loc[i, 'mri_90q'] = res['mri_90q']
        pet_mri_tab.loc[i, 'mri_95q'] = res['mri_95q']
        pet_mri_tab.loc[i, 'mri_99q'] = res['mri_99q']
        pet_mri_tab.loc[i, 'mri_999q'] = res['mri_999q']
        
pet_all_values = np.concatenate([r['pet_values'] for r in results if not r['invalid']])
mri_all_values = np.concatenate([r['mri_values'] for r in results if not r['invalid']])
i_invalid = [r['i'] for r in results if r['invalid']]

# Final min/max stats
print(f"Invalid indexes: {i_invalid}")
print(f"PET min: {pet_mri_tab['pet_min'].min()}, max: {pet_mri_tab['pet_max'].max()}")
print(f"MRI min: {pet_mri_tab['mri_min'].min()}, max: {pet_mri_tab['mri_max'].max()}")


In [1]:
# Drop invalid rows
pet_mri_tab.drop(index=i_invalid, inplace=True)
pet_mri_tab.reset_index(drop=True, inplace=True)

print(f"Remaining valid rows: {pet_mri_tab.shape[0]}")

NameError: name 'pet_mri_tab' is not defined

In [ ]:
pet_all_values = np.array(pet_all_values)
mri_all_values = np.array(mri_all_values)

p_quant90 = np.quantile(pet_all_values, 0.90)
m_quant90 = np.quantile(mri_all_values, 0.90)

p_quant95 = np.quantile(pet_all_values, 0.95)
m_quant95 = np.quantile(mri_all_values, 0.95)


p_quant99 = np.quantile(pet_all_values, 0.99)
m_quant99 = np.quantile(mri_all_values, 0.99)

p_quant999 = np.quantile(pet_all_values, 0.999)
m_quant999 = np.quantile(mri_all_values, 0.999)

p_mean = np.mean(pet_all_values)
m_mean = np.mean(mri_all_values)

p_std = np.std(pet_all_values)
m_std = np.std(mri_all_values)

p_median = np.median(pet_all_values)
m_median = np.median(mri_all_values)

p_iqr = iqr(pet_all_values, rng=(25, 75))
m_iqr = iqr(mri_all_values, rng=(25, 75))

p_after_clip = np.clip(pet_all_values, 0, p_quant999, out=None)
m_after_clip = np.clip(mri_all_values, 0, m_quant999, out=None)
p_mean_clip = np.mean(p_after_clip)
m_mean_clip = np.mean(m_after_clip)
p_std_clip = np.std(p_after_clip)
m_std_clip = np.std(m_after_clip)
p_min_clip = np.min(p_after_clip)
m_min_clip = np.min(m_after_clip)
p_max_clip = np.max(p_after_clip)
m_max_clip = np.max(m_after_clip)


np.savez("stats_train.npz",
         p_quant90=p_quant90,
         m_quant90=m_quant90,
         p_quant95=p_quant95,
         m_quant95=m_quant95,
         p_quant99=p_quant99, 
         m_quant99=m_quant99,
         p_quant999=p_quant999, 
         m_quant999=m_quant999,
         p_mean=p_mean, 
         m_mean=m_mean, 
         p_std=p_std, 
         m_std=m_std,
         p_median=p_median, 
         m_median=m_median, 
         p_iqr=p_iqr, 
         m_iqr=m_iqr,
         p_mean_clip=p_mean_clip, 
         m_mean_clip=m_mean_clip,
         p_std_clip=p_std_clip, 
         m_std_clip=m_std_clip,
         p_mim_clip=p_min_clip, 
         m_min_clip=m_min_clip,
         p_max_clip=p_max_clip, 
         m_max_clip=m_max_clip
        )


In [ ]:
plt.figure(figsize=(10, 5))

# PET Histogram
plt.subplot(2, 2, 1)
plt.hist(pet_all_values, bins=500, color='blue', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")

# MRI Histogram
plt.subplot(2, 2, 2)
plt.hist(mri_all_values, bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


plt.subplot(2, 2, 3)
plt.hist(np.log1p(pet_all_values), bins=500, color='blue', alpha=0.7, edgecolor='black')  # Log scale on Y-axis
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")



plt.subplot(2, 2, 4)
plt.hist(np.log1p(mri_all_values), bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


# Save and show the plot
plt.tight_layout()
plt.savefig("pixels_value_histograms.png")  # Save the figure
plt.show()  # Display the plot

In [ ]:
plt.figure(figsize=(10, 5))

# PET Histogram
plt.subplot(2, 2, 1)
plt.hist(p_after_clip, bins=500, color='blue', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")

# MRI Histogram
plt.subplot(2, 2, 2)
plt.hist(m_after_clip, bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


plt.subplot(2, 2, 3)
plt.hist(np.log1p(p_after_clip), bins=500, color='blue', alpha=0.7, edgecolor='black')  # Log scale on Y-axis
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("PET Image Histogram")



plt.subplot(2, 2, 4)
plt.hist(np.log1p(m_after_clip), bins=500, color='red', alpha=0.7, edgecolor='black')
plt.xlabel("Log Pixel Value")
plt.ylabel("Frequency")
plt.title("MRI Image Histogram")


# Save and show the plot
plt.tight_layout()
plt.savefig("pixels_value_histograms_clip999.png")  # Save the figure
plt.show()  # Display the plot

In [22]:
i_invalid #pet_mri_tab.drop(index = i_invalid)

[]

In [41]:
#pet_mri_tab = pet_mri_tab[pet_mri_tab.mri_min >= 0]
pet_mri_tab.reset_index(drop=True, inplace = True)

In [42]:
pet_minimum = np.min(pet_mri_tab['pet_min'])
pet_maximum = np.max(pet_mri_tab['pet_max'])
mri_minimum = np.min(pet_mri_tab['mri_min'])
mri_maximum = np.max(pet_mri_tab['mri_max'])
print(f'MinMax computation is done! PET minimum is {pet_minimum}, PET maximum is {pet_maximum}. MRI minimum is {mri_minimum}, MRI maximum is {mri_maximum}')


MinMax computation is done! PET minimum is 0.0, PET maximum is 1270.4830322265625. MRI minimum is 0.0, MRI maximum is 26299.69921875


In [32]:
pet_mri_tab.to_csv('pet_mri_pairs.csv')

In [44]:
pet_mri_tab.shape

(3471, 30)

In [45]:
suc

4983

In [46]:
er

3232

In [49]:
pet_mri_tab.Phase.value_counts()

ADNI 2     1749
ADNI GO     210
ADNI 1       24
Name: Phase, dtype: int64

In [50]:
pet_mri_tab.Project.value_counts()

ADNI    1983
A4      1488
Name: Project, dtype: int64

In [22]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 1'].MRI_Type.value_counts()

Processed    132
Name: MRI_Type, dtype: int64

In [23]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 2'].MRI_Type.value_counts()

Processed    1840
Name: MRI_Type, dtype: int64

In [24]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI GO'].MRI_Type.value_counts()

Processed    251
Name: MRI_Type, dtype: int64

In [25]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 3'].MRI_Type.value_counts()

Original    971
Name: MRI_Type, dtype: int64

In [26]:
pet_mri_tab[pet_mri_tab.Project == 'A4'].MRI_Type.value_counts()

Processed    1788
Name: MRI_Type, dtype: int64

In [165]:
pet_mri_tab.MRI_Modality.value_counts()

MRI    4982
Name: MRI_Modality, dtype: int64

In [ ]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 1'][pet_mri_tab.MRI_Type == 'Processed']

In [31]:
pet_mri_tab[pet_mri_tab.Phase == 'ADNI 3'][pet_mri_tab.MRI_Type == 'Processed']

/csc/epitkane/home/atagmazi/.conda/envs/atagmazi_gpu8/lib/python3.7/site-packages/ipykernel_launcher.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  """Entry point for launching an IPython kernel.


,Subject.ID,Project,Phase,Sex,Weight,Research.Group,VISCODE,Age,Modality,Description,Imaging.Protocol,Image.Data.ID,modality_subtype,PATH,MRI_subID,MRI_Modality,MRI_Research.Group,MRI_Age,MRI_VISCODE,MRI_Description,MRI_Imaging.protocol,MRI_Type,MRI_ID,MRI_PATH


In [33]:
pet_mri_tab[pet_mri_tab.Project == 'ADNI'].MRI_Description.value_counts()

Accelerated Sagittal MPRAGE             662
MT1; GradWarp; N3m <- MPRAGE            444
MT1; GradWarp; N3m <- MPRAGE GRAPPA2    400
MT1; N3m <- MPRAGE                      202
MT1; N3m <- MPRAGE SENSE2               180
                                       ... 
MT1; N3m <- MPRAGE REPEAT ASO             1
MT1; N3m <- ADNI SH    MPRAGE ASOX2       1
MPR; ; N3 <- ADNI SH    MPRAGE ASO        1
MT1; GradWarp; N3m <- MP-RAGE repeat      1
3D MPRAGE                                 1
Name: MRI_Description, Length: 84, dtype: int64

In [34]:
import seaborn as sns
import matplotlib.pyplot as plt

In [35]:
pet_mri_tab['Research.Group'].value_counts().values

array([1247, 1150,  722,  541,  487,  364,  251,  220])

In [36]:
pet_mri_tab['Research.Group'].value_counts()

amyloidE           1247
CN                 1150
EMCI                722
LEARN amyloidNE     541
MCI                 487
LMCI                364
AD                  251
SMC                 220
Name: Research.Group, dtype: int64

In [ ]:
plt.figure(figsize=(15,6))

ax = sns.violinplot(x="Research.Group", y="Age", data=pet_mri_tab)

# Calculate number of obs per group & median to position labels
medians = pet_mri_tab.groupby(['Research.Group'])['Age'].max().values
nobs = pet_mri_tab['Research.Group'].value_counts(sort=False).values
nobs = [str(x) for x in nobs.tolist()]
nobs = ["n: " + i for i in nobs]
 
# Add text to the figure
pos = range(len(nobs))
for tick, label in zip(pos, ax.get_xticklabels()):
   ax.text(pos[tick], medians[tick] + 0.03, nobs[tick],
            horizontalalignment='left',
            size=13,
            color='blue',
            weight='semibold')
#plt.show()
plt.savefig('samples_dist.png')

In [ ]:
pd.set_option('display.max_rows', None)
pet_mri_tab[pet_mri_tab.Project == 'A4'].Age.value_counts()

In [ ]:
pet_mri_tab[pet_mri_tab.Age == 0.0]

In [ ]:
pet_mri_tab.Age 